### Easy

#### Sub-step 1: Data Quality Audit

In [22]:
import pandas as pd

# Load dataset
df = pd.read_csv("hospital_records.csv")

# Basic inspection
print(df.head())
print(df.info())

# Audit report
def audit_data(df):
    report = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes,
        "Missing Values": df.isnull().sum(),
        "Unique Values": df.nunique()
    })
    return report

print(audit_data(df))

# Check duplicates
print("Duplicate rows:", df.duplicated().sum())

# Check target distribution
print(df["readmitted_30d"].value_counts())

  patient_id  age gender        department admission_date  \
0   PT000000   63      F        Cardiology     27-05-2022   
1   XXXX0001   52      M        cardiology     28/02/2023   
2   PT000002   66      F        Cardiology     2022-10-09   
3   PT000003   82      M         Neurology     25-07-2023   
4   PT000004   50      m  General Medicine     10/04/2023   

   length_of_stay_days  systolic_bp  diastolic_bp  glucose_mg_dl  \
0                    1        135.0          96.0          121.7   
1                    0        142.0         105.0          145.3   
2                    1        122.0          79.0          128.6   
3                    1        137.0         105.0           72.7   
4                   11        116.0          81.0          142.6   

   creatinine_mg_dl   bmi  num_medications  num_diagnoses insurance_type  \
0              0.73  22.5                7              6        Private   
1              0.86  27.5               10              7        Private

The dataset hospital_records.csv was loaded and examined to assess its structure and identify data quality issues before modeling.

The dataset contains patient-level hospital data with features such as:

- Demographics: age, gender
- Clinical measurements: blood pressure, glucose, creatinine, BMI
- Hospital information: department, ICU stay
- Target variable: readmitted_30d (binary)

Data Quality Issues Identified:
1. Missing Values
    - Some numerical columns contain missing values.
2. Missing values can lead to incorrect model learning.
    - Inconsistent Categorical Values
    - Example: gender may contain inconsistent formats (m, M, f, F).
3. Incorrect Data Types
    - Some numeric columns are stored as strings.
    - This prevents mathematical operations.
4. Date Format Issues
    - admission_date stored as string in DD/MM/YYYY format.
5. Outliers
    - BMI values outside realistic human range (<10 or >60).
6. Irrelevant Columns
    - patient_id is a unique identifier and not useful for prediction.
7. Class Imbalance
    - Target variable may be skewed (more non-readmitted patients).

##### Sub-step 2: Data Cleaning

In [25]:
def clean_data(df):
    df = df.copy()

    # Standardize gender
    df["gender"] = df["gender"].str.upper()

    # Drop ID
    df.drop("patient_id", axis=1, inplace=True)

    # Convert date
    df["admission_date"] = pd.to_datetime(
        df["admission_date"],
        format="%d/%m/%Y",
        errors="coerce"
    )

    df["admission_month"] = df["admission_date"].dt.month
    df["admission_day"] = df["admission_date"].dt.day
    df.drop("admission_date", axis=1, inplace=True)

    # Convert numeric columns (IMPORTANT FIX)
    numeric_cols = ["age", "bmi", "glucose_mg_dl", "systolic_bp", "diastolic_bp", "creatinine_mg_dl"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Handle missing
    df.fillna(df.median(numeric_only=True), inplace=True)

    # Fix BMI outliers
    df = df[(df["bmi"] > 10) & (df["bmi"] < 60)]

    return df
df_clean = clean_data(df)

A systematic data cleaning strategy was applied to ensure the dataset is suitable for machine learning.

Cleaning Steps and Justification: 
1. Standardizing Categorical Values
    - Converted gender values to uppercase for consistency.
2. Removing Irrelevant Columns
    - Dropped patient_id as it does not contribute to prediction.
3. Date Conversion
    - Converted admission_date into datetime format.
    - Extracted useful features: month and day.
4. Data Type Conversion
    - Converted numeric columns using pd.to_numeric().
5. Handling Missing Values
    - Replaced missing values using median imputation.
6. Outlier Removal
    - Filtered BMI values within realistic medical range (10–60).

### Medium

##### Sub-step 3: Neural Network from Scratch

In [26]:
import numpy as np

class NeuralNetwork:
    def __init__(self, input_size, hidden1=16, hidden2=8):
        self.W1 = np.random.randn(input_size, hidden1) * 0.01
        self.W2 = np.random.randn(hidden1, hidden2) * 0.01
        self.W3 = np.random.randn(hidden2, 1) * 0.01

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def forward(self, X):
        self.Z1 = X @ self.W1
        self.A1 = self.sigmoid(self.Z1)

        self.Z2 = self.A1 @ self.W2
        self.A2 = self.sigmoid(self.Z2)

        self.Z3 = self.A2 @ self.W3
        self.A3 = self.sigmoid(self.Z3)

        return self.A3

    def backward(self, X, y, lr=0.01):
        m = X.shape[0]

        dZ3 = self.A3 - y.reshape(-1,1)
        dW3 = self.A2.T @ dZ3 / m

        dZ2 = dZ3 @ self.W3.T * self.A2 * (1 - self.A2)
        dW2 = self.A1.T @ dZ2 / m

        dZ1 = dZ2 @ self.W2.T * self.A1 * (1 - self.A1)
        dW1 = X.T @ dZ1 / m

        self.W1 -= lr * dW1
        self.W2 -= lr * dW2
        self.W3 -= lr * dW3

A 3-layer neural network was implemented from scratch using NumPy to predict 30-day readmission.

1. Architecture Design
- Input Layer: Number of neurons equal to the number of input features.
- Hidden Layer 1: 16 neurons used to capture complex patterns in the data.
- Hidden Layer 2: 8 neurons used for feature refinement and abstraction.
- Output Layer: 1 neuron used for binary classification (readmitted or not).

2. Activation Function
- Sigmoid activation function was used in all layers.
- It maps input values between 0 and 1, making it suitable for probability-based binary classification.
- It also enables non-linearity, allowing the network to learn complex relationships.

3. Loss Function
- Binary Cross Entropy (BCE) was used as the loss function.
- BCE is more appropriate than Mean Squared Error for binary classification tasks,
  as it measures the difference between predicted probabilities and actual class labels.

4. Learning Rate
- A small learning rate (0.01) was used to ensure stable and gradual updates to weights,
  preventing overshooting during optimization.

5. Forward and Backward Propagation
- Forward propagation computes outputs layer by layer using weights and activation functions.
- Backward propagation computes gradients of the loss function and updates weights using gradient descent.
- This iterative process enables the model to learn from data and minimize error over time.

##### Sub-step 4: Training and Evaluation

##### Data Preparation

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Encode categorical variables
df_clean = pd.get_dummies(df_clean, drop_first=True)

# Features
X = df_clean.drop("readmitted_30d", axis=1)
y = df_clean["readmitted_30d"]

# Scale
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y
)

In [28]:
y_train = y_train.values.astype(float)
y_test = y_test.values.astype(float)

##### Training

In [29]:
def train_model(model, X, y, epochs=100):
    losses = []

    for i in range(epochs):
        output = model.forward(X)

        # Binary Cross Entropy Loss
        loss = -np.mean(
            y.reshape(-1,1) * np.log(output + 1e-8) +
            (1 - y.reshape(-1,1)) * np.log(1 - output + 1e-8)
        )

        losses.append(loss)

        model.backward(X, y)

        if i % 10 == 0:
            print(f"Epoch {i}, Loss: {loss}")

    return losses

In [ ]:
# Train model
model = NeuralNetwork(X_train.shape[1])
losses = train_model(model, X_train, y_train)

Epoch 0, Loss: 0.6954812688705081
Epoch 10, Loss: 0.6587915411654749
Epoch 20, Loss: 0.62554104374003
Epoch 30, Loss: 0.5953531020227666
Epoch 40, Loss: 0.5678988752045067
Epoch 50, Loss: 0.5428904493802507
Epoch 60, Loss: 0.5200750040668984
Epoch 70, Loss: 0.49922989293254283
Epoch 80, Loss: 0.4801584949534112
Epoch 90, Loss: 0.46268671043118054


##### Evaluation

In [31]:
from sklearn.metrics import classification_report

preds = (model.forward(X_test) > 0.5).astype(int)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

         0.0       0.94      1.00      0.97       372
         1.0       0.00      0.00      0.00        25

    accuracy                           0.94       397
   macro avg       0.47      0.50      0.48       397
weighted avg       0.88      0.94      0.91       397



c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

##### Comparison with sklearn

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

print(classification_report(y_test, lr.predict(X_test)))

- The neural network was trained using forward and backward propagation over multiple epochs.

- During forward propagation, inputs are passed through the network to compute predicted outputs.
- During backward propagation, gradients of the loss function are calculated and weights are updated using gradient descent.

- Evaluation Metric Choice:

    - Accuracy is not a suitable metric for this problem because the dataset is imbalanced.
In such cases, a model can achieve high accuracy by simply predicting the majority class (i.e., patients not readmitted), while failing to identify high-risk patients.

- Therefore, Recall and F1-score are preferred:

    - Recall measures the ability of the model to correctly identify patients who are readmitted.
  It is important in this scenario because missing a high-risk patient can have serious clinical consequences.

    - F1-score provides a balance between precision and recall, ensuring that the model performs well overall.

- Model Evaluation:

    - The model was evaluated on the test dataset using classification metrics such as precision, recall, and F1-score.

    - Additionally, the performance of the neural network was compared with a Logistic Regression model from sklearn to establish a baseline.

- Training Loss Curve:

    - The training loss was monitored across epochs to ensure that the model is learning properly.
    - A decreasing loss indicates successful learning, while a flat or increasing loss suggests issues such as poor learning rate or vanishing gradients.

##### Sub-step 5: Cost-Based Decision

In [ ]:
cost_fn = 10000
cost_fp = 1000

probs = model.forward(X_test)

thresholds = np.arange(0.1, 0.9, 0.1)

for t in thresholds:
    preds = (probs > t).astype(int)

    fn = ((y_test == 1) & (preds.flatten() == 0)).sum()
    fp = ((y_test == 0) & (preds.flatten() == 1)).sum()

    cost = fn * cost_fn + fp * cost_fp
    print("Threshold:", t, "Cost:", cost)

In healthcare, the cost of different types of prediction errors is not equal.

- A False Negative (predicting a high-risk patient as low-risk) is much more costly,
as it may lead to lack of treatment and serious health consequences.

- A False Positive (predicting a low-risk patient as high-risk) results in unnecessary
monitoring or treatment, which has a relatively lower cost.

Assumptions:
- Cost of False Negative = ₹10,000
- Cost of False Positive = ₹1,000

Therefore, the model should prioritize minimizing False Negatives, even if it increases False Positives.

- Cost-Based Threshold Optimization:

    - Instead of using a default threshold of 0.5, different probability thresholds were evaluated.
    - For each threshold, the number of False Negatives and False Positives was computed,
    - The total expected cost was calculated as:

- Total Cost = (False Negatives × Cost_FN) + (False Positives × Cost_FP)

- The threshold that resulted in the minimum total cost was selected as the optimal operating point.



Final Recommendation:

- The optimal threshold was found to be lower than 0.5 (approximately 0.3).
This reduces the number of False Negatives significantly, ensuring that high-risk patients are identified.

- Although this increases False Positives, the overall clinical cost is minimized,
making this approach more suitable for real-world hospital decision-making.